# LeetCode 772 — Basic Calculator III bằng Python

- Link: https://leetcode.com/problems/basic-calculator-iii/
- Độ khó: **Hard**
- Chủ đề: Ngăn xếp (Stack), Đệ quy, Phân tích cú pháp biểu thức (recursive descent, shunting-yard), Mô phỏng

> **Ghi chú (bài Premium):** đề bài dưới đây được viết lại theo hiểu biết về bài toán (các ký tự cho phép, quy tắc chia, ràng buộc). Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại phát biểu chính thức — đặc biệt là việc có hay không dấu cách và dấu trừ một ngôi. Code trong notebook xử lý được cả hai để an toàn.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def calculate(self, s: str) -> int:
```

## 1. Giới thiệu & mục tiêu học

Viết một máy tính đơn giản: biểu thức gồm số nguyên không âm, bốn phép `+ - * /` và dấu ngoặc. Bài này là **tổng hợp** của Basic Calculator I (*224*, có ngoặc, chỉ `+ -`) và Basic Calculator II (*227*, có `* /`, không ngoặc). Đây là bài kinh điển trong phỏng vấn vì kiểm tra khả năng **xử lý độ ưu tiên toán tử**, **ngoặc lồng nhau**, và **cẩn thận chi tiết** (chia lấy phần nguyên về 0, số nhiều chữ số).

Sau notebook này bạn sẽ:

1. Viết lời giải "gỡ ngoặc trong cùng" bằng thao tác chuỗi (baseline dễ hiểu).
2. Nắm kỹ thuật **ngăn xếp số hạng** của Calculator II và mở rộng bằng **đệ quy** khi gặp ngoặc.
3. Viết **bộ phân tích đệ quy đi xuống** theo văn phạm `expr/term/factor`.
4. Viết **shunting-yard hai ngăn xếp** (không đệ quy).
5. Cài đúng **phép chia cắt về 0** trong Python (khác `//`!).

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho chuỗi `s` là một biểu thức hợp lệ gồm: số nguyên không âm, các toán tử `'+'`, `'-'`, `'*'`, `'/'`, và dấu ngoặc `'('`, `')'`. Tính giá trị biểu thức. **Phép chia số nguyên cắt phần thập phân về phía 0** (ví dụ `7 / -2 = -3`). Không được dùng hàm có sẵn như `eval()`.

**Ví dụ 1**

```text
Input: s = "1+1"
Output: 2
```

**Ví dụ 2**

```text
Input: s = "6-4/2"
Output: 4
```

**Ví dụ 3**

```text
Input: s = "2*(5+5*2)/3+(6/2+8)"
Output: 21
Giải thích: 2*15 = 30; 30/3 = 10; 6/2+8 = 11; 10 + 11 = 21.
```

**Ví dụ 4** (thường gặp): `"(2+6*3+5-(3*14/7+2)*5)+3"` → `-12`.

**Ràng buộc** (theo hiểu biết)

- `1 <= s.length <= 10^4`
- `s` gồm chữ số, `'+'`, `'-'`, `'*'`, `'/'`, `'('`, `')'` (có thể có dấu cách — ta bỏ qua).
- `s` là biểu thức hợp lệ; mọi kết quả trung gian nằm trong `[-2^31, 2^31 - 1]`; không chia cho 0.

In [1]:
EXAMPLES = [
    ("1+1", 2),
    ("6-4/2", 4),
    ("2*(5+5*2)/3+(6/2+8)", 21),
    ("(2+6*3+5-(3*14/7+2)*5)+3", -12),
    ("0", 0),
]
print(EXAMPLES)

[('1+1', 2), ('6-4/2', 4), ('2*(5+5*2)/3+(6/2+8)', 21), ('(2+6*3+5-(3*14/7+2)*5)+3', -12), ('0', 0)]


## 3. Kiến thức nền

### 3.1. Chia cắt về 0 trong Python

Toán tử `//` của Python **làm tròn xuống** (về −∞): `-7 // 2 = -4`. Đề yêu cầu cắt về 0: `-7 / 2 → -3`. Hai cách an toàn:

- `int(a / b)` — dùng số thực; đúng với số trong phạm vi 32-bit (độ chính xác double đủ).
- Thuần số nguyên: `q = abs(a) // abs(b)`, đổi dấu nếu `a` và `b` trái dấu.

In [2]:
def div0(a, b):
    q = abs(a) // abs(b)
    return q if (a >= 0) == (b > 0) else -q

for a, b in [(7, 2), (-7, 2), (7, -2), (-7, -2), (0, 5), (-1, 3)]:
    print(f"{a:>3} / {b:>2}: //={a // b:>3}  int(a/b)={int(a / b):>3}  div0={div0(a, b):>3}")

  7 /  2: //=  3  int(a/b)=  3  div0=  3
 -7 /  2: //= -4  int(a/b)= -3  div0= -3
  7 / -2: //= -4  int(a/b)= -3  div0= -3
 -7 / -2: //=  3  int(a/b)=  3  div0=  3
  0 /  5: //=  0  int(a/b)=  0  div0=  0
 -1 /  3: //= -1  int(a/b)=  0  div0=  0


### 3.2. Đọc số nhiều chữ số

Đọc từng ký tự: `num = num * 10 + int(c)`. Khi gặp ký tự không phải chữ số, số đã đọc xong.

In [3]:
s = "123+45"
num, nums = 0, []
for c in s + "#":
    if c.isdigit():
        num = num * 10 + int(c)
    else:
        nums.append(num); num = 0
print(nums)

[123, 45]


### 3.3. Mẹo ngăn xếp của Calculator II (không ngoặc)

Duyệt từ trái sang, giữ `op` = toán tử **đứng trước** số hiện tại (ban đầu `'+'`). Khi đọc xong một số `num`:

- `op == '+'` → push `num`; `op == '-'` → push `-num`;
- `op == '*'` → `stack[-1] *= num`; `op == '/'` → `stack[-1] = div0(stack[-1], num)`.

Cuối cùng `sum(stack)`. Nhân/chia được làm ngay (ưu tiên cao), cộng/trừ để dành.

In [4]:
def calc2(s):
    stack, num, op = [], 0, '+'
    for c in s + '+':
        if c.isdigit():
            num = num * 10 + int(c)
        elif c in '+-*/':
            if op == '+': stack.append(num)
            elif op == '-': stack.append(-num)
            elif op == '*': stack[-1] *= num
            else: stack[-1] = div0(stack[-1], num)
            op, num = c, 0
    return sum(stack)

print(calc2("3+2*2"), calc2("14-3/2"), calc2("6-4/2"))

7 13 4


### 3.4. Văn phạm biểu thức

```text
expr   := term   (('+' | '-') term)*
term   := factor (('*' | '/') factor)*
factor := NUMBER | '(' expr ')' | '-' factor     (dòng cuối: dấu trừ một ngôi, nếu có)
```

Tầng càng sâu, ưu tiên càng cao. Vòng `while` trong mỗi tầng tạo **kết hợp trái**: `8-3-2 = (8-3)-2`.

### 3.5. `nonlocal` và biến chỉ số dùng chung

Các hàm lồng nhau cùng đọc một chuỗi cần chung một chỉ số `i`. Dùng `nonlocal i` (hoặc một list `[i]`, hoặc thuộc tính đối tượng) để cập nhật.

In [5]:
def counter_demo():
    i = 0
    def step():
        nonlocal i
        i += 1
    for _ in range(3):
        step()
    return i

print(counter_demo())

3


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Tính tay ví dụ 3: `2*(5+5*2)/3+(6/2+8)`

1. Ngoặc thứ nhất `5+5*2`: nhân trước `5*2 = 10`, rồi `5+10 = 15`.
2. Biểu thức thành `2*15/3+(6/2+8)`.
3. Ngoặc thứ hai `6/2+8 = 3+8 = 11` → `2*15/3+11`.
4. `*` và `/` cùng mức, từ trái sang: `2*15 = 30`, `30/3 = 10`. Rồi `10+11 = 21`.

### 4.2. Ngoặc = "biểu thức con độc lập"

Bên trong một cặp ngoặc là một biểu thức hoàn chỉnh. Khi tính xong, cả cặp ngoặc cư xử như **một con số**. Đây là gợi ý cho đệ quy: gặp `(` → gọi đệ quy tính biểu thức con, nhận về số, tiếp tục như bình thường.

### 4.3. Thứ tự trái sang phải với `/` cắt về 0 rất quan trọng

`2*15/3`: `(2*15)/3 = 10`, còn `2*(15/3) = 10` — trùng ở đây, nhưng `7*3/2`: `(7*3)/2 = 10` khác `7*(3/2) = 7`. Phải tính **từ trái sang**.

In [6]:
print(div0(7 * 3, 2), 7 * div0(3, 2))

10 7


## 5. Cách 1 — Brute Force: gỡ ngoặc trong cùng bằng thao tác chuỗi

**Ý tưởng.**

1. Tìm cặp ngoặc **trong cùng** (một `(` mà sau nó tới `)` không còn ngoặc nào).
2. Tính biểu thức phẳng bên trong bằng `calc2` (mở rộng cho số âm do bước trước sinh ra).
3. Thay cả cặp ngoặc bằng kết quả (dạng chuỗi). Lặp lại tới khi hết ngoặc.

Khó khăn: kết quả có thể **âm**, ví dụ `3-(5)` → `3--5`? Ta cần `calc2` hiểu dấu trừ một ngôi. Ta viết `calc_flat` dùng token hoá: một `-` đứng đầu hoặc ngay sau toán tử là dấu của số.

In [7]:
import re

def calc_flat(expr):
    """Tính biểu thức không ngoặc, cho phép số có dấu '-' một ngôi (do thay ngoặc sinh ra)."""
    toks = re.findall(r'\d+|[-+*/]', expr)
    # gộp dấu một ngôi: '-' ở đầu hoặc sau một toán tử
    vals, i, prev_is_op = [], 0, True
    while i < len(toks):
        t = toks[i]
        if prev_is_op and t in '+-':
            sign = 1
            while toks[i] in '+-':
                if toks[i] == '-':
                    sign = -sign
                i += 1
            vals.append(sign * int(toks[i])); prev_is_op = False
        elif t.isdigit():
            vals.append(int(t)); prev_is_op = False
        else:
            vals.append(t); prev_is_op = True
        i += 1
    stack, op = [], '+'
    for v in vals + ['+']:
        if isinstance(v, int):
            num = v
        else:
            if op == '+': stack.append(num)
            elif op == '-': stack.append(-num)
            elif op == '*': stack[-1] *= num
            else: stack[-1] = div0(stack[-1], num)
            op = v
    return sum(stack)

def brute(s):
    s = s.replace(' ', '')
    while '(' in s:
        m = re.search(r'\(([^()]*)\)', s)       # ngoặc trong cùng
        val = calc_flat(m.group(1))
        s = s[:m.start()] + str(val) + s[m.end():]
    return calc_flat(s)

for s, exp in EXAMPLES:
    print(f"{s:28} -> {brute(s):>4}  (kỳ vọng {exp})")

1+1                          ->    2  (kỳ vọng 2)
6-4/2                        ->    4  (kỳ vọng 4)
2*(5+5*2)/3+(6/2+8)          ->   21  (kỳ vọng 21)
(2+6*3+5-(3*14/7+2)*5)+3     ->  -12  (kỳ vọng -12)
0                            ->    0  (kỳ vọng 0)


**Độ phức tạp.** Mỗi lần gỡ một cặp ngoặc tốn `O(n)` (tìm + cắt ghép chuỗi), có tới `O(n)` cặp → `O(n^2)`. Với `n = 10^4` khoảng `10^8` thao tác ký tự — chậm, và việc ghép chuỗi số âm khá rối.

**Nút thắt.** Ta quét lại chuỗi nhiều lần. Cần một thuật toán **một lượt** xử lý ngoặc ngay khi gặp.

## 6. Bước đột phá

**Insight 1 — Ngoặc là lời gọi đệ quy.** Khi đọc tới `(`, gọi hàm tính biểu thức từ vị trí sau `(` đến `)` tương ứng; hàm trả về **giá trị** và **vị trí** sau `)`. Với người gọi, giá trị đó đóng vai một số bình thường.

**Insight 2 — Mỗi tầng ngoặc dùng mẹo ngăn xếp của Calculator II.** Bên trong một cặp ngoặc chỉ còn `+ - * /` trên các "số" (số thật hoặc kết quả ngoặc con).

**Insight 3 — Hoặc dùng văn phạm.** `expr/term/factor` tự động xử lý cả ưu tiên lẫn ngoặc; `factor` gặp `(` thì gọi `expr`.

**Insight 4 — Hoặc khử đệ quy bằng hai ngăn xếp** (shunting-yard): ngăn xếp toán tử giữ `(`, khi gặp `)` thì "xả" toán tử tới `(`.

**Bất biến (ngăn xếp số hạng).** Tại mỗi tầng, sau khi xử lý xong một số, `sum(stack)` cộng với `op` và số đang đọc mô tả đúng giá trị phần biểu thức đã đọc ở tầng đó; các phần tử của `stack` là các **hạng** đã hoàn tất phép nhân/chia, có dấu.

**Quy tắc nếu… thì…**

- Nếu gặp chữ số → tích luỹ `num`.
- Nếu gặp `(` → `num = đệ quy(...)`.
- Nếu gặp toán tử hoặc `)` hoặc hết chuỗi → áp `op` trước đó lên `num`, cập nhật `op`.
- Nếu gặp `)` → trả `sum(stack)` cho tầng trên.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Ngăn xếp số hạng + đệ quy theo ngoặc

**Giải thích.** Hàm `helper(i)` tính biểu thức bắt đầu tại `i`, dừng ở `)` khớp (hoặc cuối chuỗi), trả về `(giá trị, vị trí kế tiếp)`.

In [8]:
def calc_stack_rec(s):
    s = s.replace(' ', '')
    n = len(s)

    def helper(i):
        stack, num, op = [], 0, '+'
        while i <= n:
            c = s[i] if i < n else ')'          # coi cuối chuỗi như ')'
            if c.isdigit():
                num = num * 10 + int(c)
            elif c == '(':
                num, i = helper(i + 1)          # i trỏ tới ')' khớp
            if c in '+-*/)':
                if op == '+': stack.append(num)
                elif op == '-': stack.append(-num)
                elif op == '*': stack[-1] *= num
                else: stack[-1] = div0(stack[-1], num)
                if c == ')':
                    return sum(stack), i
                op, num = c, 0
            i += 1
        return sum(stack), i

    return helper(0)[0]

for s, exp in EXAMPLES:
    print(calc_stack_rec(s), exp)

2 2
4 4
21 21
-12 -12
0 0


Chú ý: sau `num, i = helper(i + 1)`, `i` trỏ đúng vào `)` của ngoặc con; câu lệnh `i += 1` cuối vòng bỏ qua nó. Ký tự `c` lúc này vẫn là `'('` nên không rơi vào nhánh toán tử — đúng ý.

**Code nộp LeetCode (Cách 2).**

In [9]:
class SolutionStackRec:
    def calculate(self, s: str) -> int:
        s = s.replace(' ', '')
        n = len(s)

        def div0(a, b):
            q = abs(a) // abs(b)
            return q if (a >= 0) == (b > 0) else -q

        def helper(i):
            stack, num, op = [], 0, '+'
            while i <= n:
                c = s[i] if i < n else ')'
                if c.isdigit():
                    num = num * 10 + int(c)
                elif c == '(':
                    num, i = helper(i + 1)
                if c in '+-*/)':
                    if op == '+':
                        stack.append(num)
                    elif op == '-':
                        stack.append(-num)
                    elif op == '*':
                        stack[-1] *= num
                    else:
                        stack[-1] = div0(stack[-1], num)
                    if c == ')':
                        return sum(stack), i
                    op, num = c, 0
                i += 1
            return sum(stack), i

        return helper(0)[0]

print([SolutionStackRec().calculate(s) for s, _ in EXAMPLES])

[2, 4, 21, -12, 0]


**Độ phức tạp.** Mỗi ký tự được xử lý một lần → `O(n)`. Bộ nhớ `O(n)` (ngăn xếp + độ sâu đệ quy bằng độ sâu ngoặc).

### 7.2. Cách 3 — Đệ quy đi xuống theo văn phạm

**Giải thích.** Ba hàm `expr`, `term`, `factor` dùng chung chỉ số `i`. `factor` xử lý số, ngoặc, và (để an toàn) dấu trừ/cộng một ngôi.

In [10]:
def calc_rd(s):
    s = s.replace(' ', '')
    n = len(s)
    i = 0

    def expr():
        nonlocal i
        v = term()
        while i < n and s[i] in '+-':
            op = s[i]; i += 1
            w = term()
            v = v + w if op == '+' else v - w
        return v

    def term():
        nonlocal i
        v = factor()
        while i < n and s[i] in '*/':
            op = s[i]; i += 1
            w = factor()
            v = v * w if op == '*' else div0(v, w)
        return v

    def factor():
        nonlocal i
        if s[i] == '(':
            i += 1
            v = expr()
            i += 1                      # bỏ ')'
            return v
        if s[i] in '+-':                # dấu một ngôi (phòng hờ)
            sign = -1 if s[i] == '-' else 1
            i += 1
            return sign * factor()
        num = 0
        while i < n and s[i].isdigit():
            num = num * 10 + int(s[i]); i += 1
        return num

    return expr()

for s, exp in EXAMPLES:
    print(calc_rd(s), exp)

2 2
4 4
21 21
-12 -12
0 0


**Code nộp LeetCode (Cách 3).**

In [11]:
class SolutionRD:
    def calculate(self, s: str) -> int:
        s = s.replace(' ', '')
        n = len(s)
        self.i = 0

        def div0(a, b):
            q = abs(a) // abs(b)
            return q if (a >= 0) == (b > 0) else -q

        def expr():
            v = term()
            while self.i < n and s[self.i] in '+-':
                op = s[self.i]; self.i += 1
                w = term()
                v = v + w if op == '+' else v - w
            return v

        def term():
            v = factor()
            while self.i < n and s[self.i] in '*/':
                op = s[self.i]; self.i += 1
                w = factor()
                v = v * w if op == '*' else div0(v, w)
            return v

        def factor():
            if s[self.i] == '(':
                self.i += 1
                v = expr()
                self.i += 1
                return v
            if s[self.i] in '+-':
                sign = -1 if s[self.i] == '-' else 1
                self.i += 1
                return sign * factor()
            num = 0
            while self.i < n and s[self.i].isdigit():
                num = num * 10 + ord(s[self.i]) - 48
                self.i += 1
            return num

        return expr()

print([SolutionRD().calculate(s) for s, _ in EXAMPLES])

[2, 4, 21, -12, 0]


**Độ phức tạp.** `O(n)` thời gian; độ sâu đệ quy `O(độ sâu ngoặc)`. Với `n = 10^4` và ngoặc lồng rất sâu (ví dụ 5000 tầng), Python có thể chạm giới hạn đệ quy → cần `sys.setrecursionlimit` hoặc dùng Cách 4.

### 7.3. Cách 4 — Shunting-yard: hai ngăn xếp, không đệ quy

**Giải thích.** `nums` chứa số, `ops` chứa toán tử và `(`.

- Số → đọc trọn, push `nums`.
- `(` → push `ops`.
- `)` → thực hiện toán tử tới khi gặp `(`, pop `(`.
- Toán tử `o` → trong khi đỉnh `ops` là toán tử có ưu tiên `>=` `o`: thực hiện; rồi push `o`.
- Hết chuỗi → thực hiện hết.

Để hỗ trợ dấu trừ một ngôi (nếu có): khi `-`/`+` xuất hiện ở đầu hoặc ngay sau `(`, push thêm số `0` trước (biến `-x` thành `0-x`).

In [12]:
PREC = {'+': 1, '-': 1, '*': 2, '/': 2}

def apply(nums, op):
    b = nums.pop(); a = nums.pop()
    if op == '+': nums.append(a + b)
    elif op == '-': nums.append(a - b)
    elif op == '*': nums.append(a * b)
    else: nums.append(div0(a, b))

def calc_shunting(s):
    s = s.replace(' ', '')
    nums, ops = [], []
    i, n = 0, len(s)
    while i < n:
        c = s[i]
        if c.isdigit():
            num = 0
            while i < n and s[i].isdigit():
                num = num * 10 + int(s[i]); i += 1
            nums.append(num)
            continue
        if c == '(':
            ops.append(c)
        elif c == ')':
            while ops[-1] != '(':
                apply(nums, ops.pop())
            ops.pop()
        else:
            if c in '+-' and (i == 0 or s[i - 1] == '('):
                nums.append(0)                       # dấu một ngôi
            while ops and ops[-1] != '(' and PREC[ops[-1]] >= PREC[c]:
                apply(nums, ops.pop())
            ops.append(c)
        i += 1
    while ops:
        apply(nums, ops.pop())
    return nums[-1]

for s, exp in EXAMPLES:
    print(calc_shunting(s), exp)

2 2
4 4
21 21
-12 -12
0 0


**Code nộp LeetCode (Cách 4).**

In [13]:
class SolutionShunting:
    def calculate(self, s: str) -> int:
        s = s.replace(' ', '')
        prec = {'+': 1, '-': 1, '*': 2, '/': 2}
        nums, ops = [], []

        def apply():
            op = ops.pop()
            b = nums.pop(); a = nums.pop()
            if op == '+':
                nums.append(a + b)
            elif op == '-':
                nums.append(a - b)
            elif op == '*':
                nums.append(a * b)
            else:
                q = abs(a) // abs(b)
                nums.append(q if (a >= 0) == (b > 0) else -q)

        i, n = 0, len(s)
        while i < n:
            c = s[i]
            if c.isdigit():
                j = i
                while j < n and s[j].isdigit():
                    j += 1
                nums.append(int(s[i:j]))
                i = j
                continue
            if c == '(':
                ops.append(c)
            elif c == ')':
                while ops[-1] != '(':
                    apply()
                ops.pop()
            else:
                if c in '+-' and (i == 0 or s[i - 1] == '('):
                    nums.append(0)
                while ops and ops[-1] != '(' and prec[ops[-1]] >= prec[c]:
                    apply()
                ops.append(c)
            i += 1
        while ops:
            apply()
        return nums[-1]

print([SolutionShunting().calculate(s) for s, _ in EXAMPLES])

[2, 4, 21, -12, 0]


**Độ phức tạp.** Mỗi số/toán tử vào và ra ngăn xếp đúng một lần → `O(n)` thời gian, `O(n)` bộ nhớ, **không đệ quy**.

### 7.4. Cách 5 — Một ngăn xếp, lưu trạng thái `(stack, op)` khi gặp ngoặc (khử đệ quy Cách 2)

**Giải thích.** Thay vì gọi đệ quy khi gặp `(`, ta **cất** trạng thái hiện tại (`stack` và `op` của tầng ngoài) vào một ngăn xếp `saved`, rồi bắt đầu tầng mới rỗng. Khi gặp `)`, chốt số cuối, tính `sum(stack)` thành `num`, rồi **khôi phục** tầng ngoài. Đây là cách khử đệ quy "thủ công" — hữu ích khi ngoặc lồng rất sâu.

In [14]:
def calc_saved(s):
    s = s.replace(' ', '') + '+'
    saved = []
    stack, num, op = [], 0, '+'

    def push(stack, op, num):
        if op == '+': stack.append(num)
        elif op == '-': stack.append(-num)
        elif op == '*': stack[-1] *= num
        else: stack[-1] = div0(stack[-1], num)

    for c in s:
        if c.isdigit():
            num = num * 10 + int(c)
        elif c == '(':
            saved.append((stack, op))
            stack, num, op = [], 0, '+'
        elif c == ')':
            push(stack, op, num)
            num = sum(stack)
            stack, op = saved.pop()
        else:                                   # + - * /
            push(stack, op, num)
            op, num = c, 0
    return sum(stack)

for s, exp in EXAMPLES:
    print(calc_saved(s), exp)

2 2
4 4
21 21
-12 -12
0 0


**Code nộp LeetCode (Cách 5).**

In [15]:
class SolutionSaved:
    def calculate(self, s: str) -> int:
        saved = []
        stack, num, op = [], 0, '+'
        for c in s.replace(' ', '') + '+':
            if c.isdigit():
                num = num * 10 + ord(c) - 48
            elif c == '(':
                saved.append((stack, op))
                stack, num, op = [], 0, '+'
            else:
                if op == '+':
                    stack.append(num)
                elif op == '-':
                    stack.append(-num)
                elif op == '*':
                    stack[-1] *= num
                else:
                    q = abs(stack[-1]) // abs(num)
                    stack[-1] = q if (stack[-1] >= 0) == (num > 0) else -q
                if c == ')':
                    num = sum(stack)
                    stack, op = saved.pop()
                else:
                    op, num = c, 0
        return sum(stack)

print([SolutionSaved().calculate(s) for s, _ in EXAMPLES])

[2, 4, 21, -12, 0]


**Độ phức tạp.** `O(n)` thời gian, `O(n)` bộ nhớ, không đệ quy.

## 8. TRACE từng bước

Trace **Cách 5** trên `"2*(5+5*2)/3+(6/2+8)"`.

| ký tự | hành động | stack | op | num | saved |
|---|---|---|---|---|---|
| `2` | num=2 | [] | + | 2 | |
| `*` | push +2 | [2] | * | 0 | |
| `(` | cất ([2], *) | [] | + | 0 | [([2],*)] |
| `5` `+` | push 5 | [5] | + | 0 | |
| `5` `*` | push 5 | [5,5] | * | 0 | |
| `2` `)` | 5*2 → [5,10]; num=15; khôi phục | [2] | * | 15 | [] |
| `/` | 2*15=30 | [30] | / | 0 | |
| `3` `+` | 30/3=10 | [10] | + | 0 | |
| `(`…`)` | tầng con 6/2+8 = 11 | [10] | + | 11 | |
| cuối | push 11 | [10,11] | | | → 21 |

In [16]:
def trace_saved(s):
    s = s.replace(' ', '') + '+'
    saved = []
    stack, num, op = [], 0, '+'
    def push():
        if op == '+': stack.append(num)
        elif op == '-': stack.append(-num)
        elif op == '*': stack[-1] *= num
        else: stack[-1] = div0(stack[-1], num)
    for idx, c in enumerate(s):
        note = ""
        if c.isdigit():
            num = num * 10 + int(c)
        elif c == '(':
            saved.append((stack, op)); note = f"cất ({stack}, {op!r})"
            stack, num, op = [], 0, '+'
        elif c == ')':
            push(); num = sum(stack)
            note = f"đóng ngoặc: tầng con = {num}"
            stack, op = saved.pop()
        else:
            push(); op, num = c, 0
        shown = c if idx < len(s) - 1 else '(hết)'
        print(f"{shown:>5} | stack={str(stack):<10} op={op!r} num={num:<3} saved={[x[0] for x in saved]} {note}")
    print("kết quả:", sum(stack))

trace_saved("2*(5+5*2)/3+(6/2+8)")

    2 | stack=[]         op='+' num=2   saved=[] 
    * | stack=[2]        op='*' num=0   saved=[] 
    ( | stack=[]         op='+' num=0   saved=[[2]] cất ([2], '*')
    5 | stack=[]         op='+' num=5   saved=[[2]] 
    + | stack=[5]        op='+' num=0   saved=[[2]] 
    5 | stack=[5]        op='+' num=5   saved=[[2]] 
    * | stack=[5, 5]     op='*' num=0   saved=[[2]] 
    2 | stack=[5, 5]     op='*' num=2   saved=[[2]] 
    ) | stack=[2]        op='*' num=15  saved=[] đóng ngoặc: tầng con = 15
    / | stack=[30]       op='/' num=0   saved=[] 
    3 | stack=[30]       op='/' num=3   saved=[] 
    + | stack=[10]       op='+' num=0   saved=[] 
    ( | stack=[]         op='+' num=0   saved=[[10]] cất ([10], '+')
    6 | stack=[]         op='+' num=6   saved=[[10]] 
    / | stack=[6]        op='/' num=0   saved=[[10]] 
    2 | stack=[6]        op='/' num=2   saved=[[10]] 
    + | stack=[3]        op='+' num=0   saved=[[10]] 
    8 | stack=[3]        op='+' num=8   saved=[[10]] 
    

Trace **Cách 4 (shunting-yard)** trên `"(1+2)*3-4/2"`:

In [17]:
def trace_shunting(s):
    nums, ops = [], []
    i, n = 0, len(s)
    while i < n:
        c = s[i]
        acts = []
        if c.isdigit():
            j = i
            while j < n and s[j].isdigit():
                j += 1
            nums.append(int(s[i:j])); tok = s[i:j]; i = j
        else:
            tok = c
            if c == '(':
                ops.append(c)
            elif c == ')':
                while ops[-1] != '(':
                    op = ops[-1]; apply(nums, ops.pop()); acts.append(f"thực hiện {op}")
                ops.pop()
            else:
                while ops and ops[-1] != '(' and PREC[ops[-1]] >= PREC[c]:
                    op = ops[-1]; apply(nums, ops.pop()); acts.append(f"thực hiện {op}")
                ops.append(c)
            i += 1
        print(f"token {tok:>2} | {', '.join(acts):<26} | nums={nums} ops={ops}")
    while ops:
        op = ops[-1]; apply(nums, ops.pop()); print(f"cuối: thực hiện {op} -> nums={nums}")
    return nums[-1]

trace_shunting("(1+2)*3-4/2")

token  ( |                            | nums=[] ops=['(']
token  1 |                            | nums=[1] ops=['(']
token  + |                            | nums=[1] ops=['(', '+']
token  2 |                            | nums=[1, 2] ops=['(', '+']
token  ) | thực hiện +                | nums=[3] ops=[]
token  * |                            | nums=[3] ops=['*']
token  3 |                            | nums=[3, 3] ops=['*']
token  - | thực hiện *                | nums=[9] ops=['-']
token  4 |                            | nums=[9, 4] ops=['-']
token  / |                            | nums=[9, 4] ops=['-', '/']
token  2 |                            | nums=[9, 4, 2] ops=['-', '/']
cuối: thực hiện / -> nums=[9, 2]
cuối: thực hiện - -> nums=[7]


7

## 9. Code nộp LeetCode

Bản chọn: **Cách 5** — một lượt, không đệ quy (an toàn với ngoặc lồng sâu), dựa trên mẹo ngăn xếp quen thuộc của Calculator II.

In [18]:
class Solution:
    def calculate(self, s: str) -> int:
        saved = []                          # các tầng ngoài: (stack, op)
        stack, num, op = [], 0, '+'         # tầng hiện tại
        for c in s.replace(' ', '') + '+':  # '+' cuối để chốt số cuối cùng
            if c.isdigit():
                num = num * 10 + ord(c) - 48
            elif c == '(':
                saved.append((stack, op))   # cất tầng ngoài
                stack, num, op = [], 0, '+'
            else:                           # c là + - * / hoặc ')'
                if op == '+':
                    stack.append(num)
                elif op == '-':
                    stack.append(-num)
                elif op == '*':
                    stack[-1] *= num
                else:                       # chia cắt về 0
                    q = abs(stack[-1]) // abs(num)
                    stack[-1] = q if (stack[-1] >= 0) == (num > 0) else -q
                if c == ')':
                    num = sum(stack)        # cả ngoặc thành một số
                    stack, op = saved.pop() # khôi phục tầng ngoài
                else:
                    op, num = c, 0
        return sum(stack)

for s, exp in EXAMPLES:
    assert Solution().calculate(s) == exp
print("OK")

OK


Giải thích những điểm tinh tế:

- **`+ '+'` ở cuối:** ký tự giả để "chốt" số cuối theo `op` hiện tại.
- **Gặp `)`:** trước tiên chốt `num` vào `stack` theo `op` của tầng con, rồi `num = sum(stack)`. Sau khi khôi phục tầng ngoài, `num` (giá trị ngoặc) sẽ được chốt khi gặp toán tử kế tiếp — đúng như một số bình thường.
- **Gặp `(` ngay sau `*`:** `op = '*'` được cất cùng `stack`; khi ngoặc đóng, `op` khôi phục là `'*'` → nhân đúng.
- **Chia:** dùng số nguyên, không dùng `//` trực tiếp trên số âm.

## 10. Vì sao thuật toán đúng

**Bổ đề (Calculator II).** Với biểu thức không ngoặc `x0 o1 x1 o2 ... ok xk`, sau khi xử lý hết, `sum(stack)` bằng giá trị biểu thức. Chứng minh: biểu thức = tổng các **hạng** (dãy con lớn nhất nối bằng `*`/`/`), mỗi hạng mang dấu của `+`/`-` đứng trước. Ngăn xếp giữ đúng mỗi hạng một phần tử (có dấu); `*`/`/` cập nhật phần tử cuối **từ trái sang**, đúng thứ tự kết hợp trái. (Lưu ý: với hạng âm như `-7/2`, ta lưu `-7` rồi chia cắt về 0 cho `-3` — đúng bằng `-(7/2)` vì cắt về 0 đối xứng.)

**Ngoặc.** Bằng quy nạp theo độ sâu ngoặc: tầng trong cùng không có ngoặc nên đúng theo bổ đề; khi `)` đến, `num` = giá trị đúng của ngoặc. Với tầng ngoài, ngoặc đã đóng là một "số" bình thường; trạng thái `(stack, op)` được khôi phục y nguyên như trước khi gặp `(`. Vậy tầng ngoài cũng thoả bổ đề.

**Cách 3** đúng vì văn phạm `expr/term/factor` sinh đúng cây cú pháp với ưu tiên và kết hợp trái. **Cách 4** tương đương Cách 3 (thuật toán shunting-yard của Dijkstra).

## 11. Phân tích Time/Space

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Gỡ ngoặc chuỗi | `O(n^2)` | `O(n)` | quét lại nhiều lần |
| 2. Ngăn xếp + đệ quy | `O(n)` | `O(n)` | đệ quy theo độ sâu ngoặc |
| 3. Đệ quy đi xuống | `O(n)` | `O(n)` | đệ quy theo độ sâu ngoặc |
| 4. Shunting-yard | `O(n)` | `O(n)` | không đệ quy |
| 5. Cất trạng thái | `O(n)` | `O(n)` | không đệ quy |

In [19]:
import random, time, sys
random.seed(772)

def rand_expr(depth=0, max_depth=4):
    if depth >= max_depth or random.random() < 0.25:
        return str(random.randint(0, 20))
    if random.random() < 0.3:
        return "(" + rand_expr(depth + 1, max_depth) + ")"
    return rand_expr(depth + 1, max_depth) + random.choice("+-*/") + rand_expr(depth + 1, max_depth)

def safe(f, s):
    try:
        return f(s)
    except ZeroDivisionError:
        return "div0"

long_expr = "+".join("(" + rand_expr(0, 3) + ")" for _ in range(200))
while safe(calc_saved, long_expr) == "div0":
    long_expr = "+".join("(" + rand_expr(0, 3) + ")" for _ in range(200))
print("độ dài:", len(long_expr))
for name, f in [("brute", brute), ("stack_rec", calc_stack_rec), ("rd", calc_rd),
                ("shunting", calc_shunting), ("saved", calc_saved)]:
    t0 = time.perf_counter()
    r = f(long_expr)
    print(f"{name:10} {r:>14}  {1000 * (time.perf_counter() - t0):7.2f} ms")

độ dài: 2313
brute               25093     1.42 ms
stack_rec           25093     0.49 ms
rd                  25093     0.41 ms
shunting            25093     0.58 ms
saved               25093     0.32 ms


## 12. Bộ test

In [20]:
methods = {
    "brute": brute, "stack_rec": calc_stack_rec, "rd": calc_rd,
    "shunting": calc_shunting, "saved": calc_saved,
    "SolutionStackRec": lambda s: SolutionStackRec().calculate(s),
    "SolutionRD": lambda s: SolutionRD().calculate(s),
    "SolutionShunting": lambda s: SolutionShunting().calculate(s),
    "SolutionSaved": lambda s: SolutionSaved().calculate(s),
    "Solution": lambda s: Solution().calculate(s),
}

# 12.1 Ví dụ
for name, f in methods.items():
    for s, exp in EXAMPLES:
        assert f(s) == exp, (name, s)

# 12.2 Edge cases
edge = [
    ("42", 42),
    ("(((7)))", 7),
    ("1-1-1", -1),                 # kết hợp trái
    ("8/3/2", 1),                  # (8/3)/2 = 2/2 = 1
    ("7*3/2", 10),                 # (7*3)/2
    ("1-7/2", -2),                 # 1 - 3
    ("(1-8)/2", -3),               # -7/2 cắt về 0 = -3, không phải -4
    ("2-(5-6)", 3),
    ("0*(3+4)", 0),
    (" 3 + 5 / 2 ", 5),            # có dấu cách
    ("2147483647", 2147483647),
    ("10/(2-3)", -10),
    ("(2-9)/(1-3)", 3),            # -7 / -2 = 3
]
for name, f in methods.items():
    for s, exp in edge:
        assert f(s) == exp, (name, s, f(s))
print("Ví dụ + edge cases: OK")

Ví dụ + edge cases: OK


In [21]:
# 12.3 Ngẫu nhiên: đối chiếu với brute force và với tham chiếu bằng cây cú pháp của Python (ast)
import ast

def ref_eval(s):
    """Tham chiếu độc lập: dùng bộ phân tích của Python (ast), tự tính với phép chia cắt về 0."""
    tree = ast.parse(s, mode='eval')
    def ev(node):
        if isinstance(node, ast.Expression):
            return ev(node.body)
        if isinstance(node, ast.Constant):
            return node.value
        if isinstance(node, ast.UnaryOp):
            return -ev(node.operand) if isinstance(node.op, ast.USub) else ev(node.operand)
        a, b = ev(node.left), ev(node.right)
        if isinstance(node.op, ast.Add): return a + b
        if isinstance(node.op, ast.Sub): return a - b
        if isinstance(node.op, ast.Mult): return a * b
        return div0(a, b)
    return ev(tree)

cnt = 0
for _ in range(1500):
    s = rand_expr()
    r = safe(ref_eval, s)
    if r == "div0":
        continue
    cnt += 1
    for name, f in methods.items():
        assert f(s) == r, (name, s, f(s), r)
print("Đã đối chiếu", cnt, "biểu thức ngẫu nhiên")

# 12.4 Ngoặc lồng rất sâu: chỉ các cách không đệ quy
deep = "(" * 3000 + "1+2*3" + ")" * 3000
assert calc_shunting(deep) == calc_saved(deep) == Solution().calculate(deep) == 7
print("Ngoặc lồng 3000 tầng: OK (Cách 4, 5)")

Đã đối chiếu 1440 biểu thức ngẫu nhiên
Ngoặc lồng 3000 tầng: OK (Cách 4, 5)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `//` cho phép chia

**Code sai:**

In [22]:
def wrong_floor_div(s):
    stack, num, op = [], 0, '+'
    for c in s + '+':
        if c.isdigit():
            num = num * 10 + int(c)
        elif c in '+-*/':
            if op == '+': stack.append(num)
            elif op == '-': stack.append(-num)
            elif op == '*': stack[-1] *= num
            else: stack[-1] //= num            # SAI với số âm
            op, num = c, 0
    return sum(stack)

print("sai:", wrong_floor_div("1-7/2"), " đúng:", calc_saved("1-7/2"))

sai: -3  đúng: -2


**Vì sao sai:** hạng `-7/2` được lưu là `-7`, `-7 // 2 = -4` (làm tròn xuống) thay vì `-3` (cắt về 0).

**Code đúng:** `div0` (hoặc `int(a / b)`).

### Lỗi 2 — Quên chốt số cuối cùng

**Code sai:**

In [23]:
def wrong_no_flush(s):
    stack, num, op = [], 0, '+'
    for c in s:                                  # SAI: không thêm ký tự chốt
        if c.isdigit():
            num = num * 10 + int(c)
        else:
            if op == '+': stack.append(num)
            elif op == '-': stack.append(-num)
            elif op == '*': stack[-1] *= num
            else: stack[-1] = div0(stack[-1], num)
            op, num = c, 0
    return sum(stack)

print("sai:", wrong_no_flush("3+2*2"), " đúng:", calc_saved("3+2*2"))

sai: 5  đúng: 7


**Vì sao sai:** số cuối (`2`) và phép `*` trước nó không bao giờ được áp dụng.

**Code đúng:** duyệt `s + '+'`, hoặc xử lý khi `i == len(s) - 1`.

### Lỗi 3 — Đánh giá thuần trái sang phải (bỏ qua ưu tiên)

**Code sai:**

In [24]:
def wrong_left_to_right(s):
    toks = re.findall(r'\d+|[-+*/]', s)
    v = int(toks[0])
    for op, x in zip(toks[1::2], toks[2::2]):
        x = int(x)
        v = v + x if op == '+' else v - x if op == '-' else v * x if op == '*' else div0(v, x)
    return v

print("sai:", wrong_left_to_right("6-4/2"), " đúng:", calc_saved("6-4/2"))

sai: 1  đúng: 4


**Vì sao sai:** `6-4/2` bị tính thành `(6-4)/2 = 1` thay vì `6-(4/2) = 4`.

**Code đúng:** mẹo ngăn xếp hạng, văn phạm, hoặc shunting-yard với độ ưu tiên.

### Lỗi 4 — Shunting-yard dùng `>` thay vì `>=` (kết hợp phải)

**Code sai:**

In [25]:
def wrong_assoc(s):
    nums, ops = [], []
    i = 0
    while i < len(s):
        c = s[i]
        if c.isdigit():
            j = i
            while j < len(s) and s[j].isdigit():
                j += 1
            nums.append(int(s[i:j])); i = j; continue
        if c == '(':
            ops.append(c)
        elif c == ')':
            while ops[-1] != '(':
                apply(nums, ops.pop())
            ops.pop()
        else:
            while ops and ops[-1] != '(' and PREC[ops[-1]] > PREC[c]:   # SAI
                apply(nums, ops.pop())
            ops.append(c)
        i += 1
    while ops:
        apply(nums, ops.pop())
    return nums[-1]

for s in ["8-3-2", "8/3/2"]:
    print(s, "sai:", wrong_assoc(s), " đúng:", calc_saved(s))

8-3-2 sai: 7  đúng: 3
8/3/2 sai: 8  đúng: 1


**Vì sao sai:** `8-3-2` thành `8-(3-2) = 7` thay vì `3`; `8/3/2` thành `8/(3/2) = 8` thay vì `1`.

**Code đúng:** `PREC[ops[-1]] >= PREC[c]`.

### Lỗi 5 — Cất ngoặc nhưng quên cất `op`

In [26]:
def wrong_forget_op(s):
    saved = []
    stack, num, op = [], 0, '+'
    for c in s + '+':
        if c.isdigit():
            num = num * 10 + int(c)
        elif c == '(':
            saved.append(stack)                   # SAI: không cất op
            stack, num, op = [], 0, '+'
        else:
            if op == '+': stack.append(num)
            elif op == '-': stack.append(-num)
            elif op == '*': stack[-1] *= num
            else: stack[-1] = div0(stack[-1], num)
            if c == ')':
                num = sum(stack); stack = saved.pop()   # op vẫn là op của tầng con!
            else:
                op, num = c, 0
    return sum(stack)

print("sai:", wrong_forget_op("2*(3+4)"), " đúng:", calc_saved("2*(3+4)"))

sai: 9  đúng: 14


**Vì sao sai:** sau khi đóng ngoặc, `op` còn là `'+'` (của tầng con) thay vì `'*'` → `2*(3+4)` thành `2+7 = 9`.

**Code đúng:** cất và khôi phục **cặp** `(stack, op)`.

## 14. Mẹo & nhận dạng pattern

- **Biểu thức `+ - * /` không ngoặc** → ngăn xếp hạng với `op` đứng trước (*227*).
- **Có ngoặc** → đệ quy khi gặp `(`, hoặc cất trạng thái `(stack, op)`.
- **Tổng quát nhiều mức ưu tiên / toán tử mới** → văn phạm hoặc shunting-yard với bảng `PREC`.
- **Chia cắt về 0** → không dùng `//` trên số âm.
- **Kiểm thử bộ phân tích** → sinh biểu thức ngẫu nhiên, so với tham chiếu độc lập (ở đây `ast` của Python).
- Họ bài: *150. Evaluate Reverse Polish Notation*, *224*, *227*, *770*, *1597. Build Binary Expression Tree From Infix Expression*.

## 15. Biến thể

1. **Có dấu trừ một ngôi** (`"-(2+3)"`, `"2*-3"`) → `factor := '-' factor` (Cách 3 đã hỗ trợ).
2. **Thêm lũy thừa `^`** (kết hợp phải) → ưu tiên 3, so sánh `>` thay vì `>=` **chỉ cho `^`**.
3. **Chuyển sang ký pháp Ba Lan ngược (RPN)** → shunting-yard xuất toán tử thay vì thực hiện.
4. **Có biến** → *770. Basic Calculator IV*.
5. **Dựng cây biểu thức** → *1597*.

In [27]:
# Biến thể 3: chuyển sang RPN rồi tính RPN (LeetCode 150)
def to_rpn(s):
    out, ops = [], []
    for tok in re.findall(r'\d+|[-+*/()]', s):
        if tok.isdigit():
            out.append(tok)
        elif tok == '(':
            ops.append(tok)
        elif tok == ')':
            while ops[-1] != '(':
                out.append(ops.pop())
            ops.pop()
        else:
            while ops and ops[-1] != '(' and PREC[ops[-1]] >= PREC[tok]:
                out.append(ops.pop())
            ops.append(tok)
    return out + ops[::-1]

def eval_rpn(tokens):
    st = []
    for t in tokens:
        if t.isdigit():
            st.append(int(t))
        else:
            apply(st, t)
    return st[-1]

rpn = to_rpn("2*(5+5*2)/3+(6/2+8)")
print(' '.join(rpn), "=", eval_rpn(rpn))

2 5 5 2 * + * 3 / 6 2 / 8 + + = 21


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Gỡ ngoặc chuỗi | Tính ngoặc trong cùng, thay bằng số | `O(n^2)` | `O(n)` | Minh hoạ, kiểm thử |
| 2. Ngăn xếp + đệ quy | Calculator II mỗi tầng, đệ quy khi `(` | `O(n)` | `O(n)` | Phổ biến trong phỏng vấn |
| 3. Đệ quy đi xuống | `expr/term/factor` | `O(n)` | `O(n)` | Dễ mở rộng toán tử/dấu một ngôi |
| 4. Shunting-yard | Hai ngăn xếp, bảng ưu tiên | `O(n)` | `O(n)` | Không đệ quy, tổng quát |
| 5. Cất trạng thái | Khử đệ quy Cách 2 | `O(n)` | `O(n)` | Nộp bài, ngoặc lồng sâu |

## 17. Bài tập

**Bài 1 (đoán output).** `calculate("(6-14)/3*2+1")` = ?

**Bài 2 (tính tay).** Viết trạng thái `stack, op, num, saved` của Cách 5 cho `"3*(2-(1+4))"`.

**Bài 3 (điền TODO).** Hoàn thiện `calc_ii(s)` (Basic Calculator II: không ngoặc, có dấu cách).

**Bài 4 (tự viết lại).** Viết lại Cách 3 để hỗ trợ thêm toán tử `%` (lấy dư, cùng mức với `*` `/`, dấu theo số bị chia như C: `a - b * trunc(a/b)`).

**Bài 5 (mini challenge).** Thêm lũy thừa `^` (ưu tiên cao nhất, kết hợp phải) vào shunting-yard.

In [28]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def calc_ii(s):
    # TODO: mẹo ngăn xếp hạng, bỏ qua dấu cách
    pass

# Bài 4 — TODO
def calc_rd_mod(s):
    # TODO: trong term(), thêm nhánh '%' -> v - w * div0(v, w)
    pass

# Bài 5 — TODO
def calc_pow(s):
    # TODO: PREC['^'] = 3; với '^' chỉ pop khi PREC[top] > 3 (kết hợp phải)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `(6-14) = -8`.
- Mức 2: `-8/3` cắt về 0 = `-2`; `-2*2 = -4`.
- Mức 3: `-4+1 = -3`.

**Bài 2.**
- Mức 1: `3*` → stack `[3]`, op `*`; gặp `(` → cất `([3], '*')`.
- Mức 2: tầng 2: `2-` → `[2]`, op `-`; gặp `(` → cất `([2], '-')`; tầng 3: `1+4 = 5`.
- Mức 3: đóng → num 5, khôi phục `[2]`, `-` → `[2,-5]` → đóng → num −3 → `[3]*(-3)` = `-9`.

**Bài 3.**
- Mức 1: bỏ dấu cách trước.
- Mức 2: duyệt `s + '+'`.
- Mức 3: dùng `div0`.

**Bài 4.**
- Mức 1: `'%'` cùng vòng `while` với `'*'`, `'/'`.
- Mức 2: dư kiểu C: `a - b * div0(a, b)`.
- Mức 3: kiểm tra `7 % -3 = 1`, `-7 % 3 = -1`.

**Bài 5.**
- Mức 1: thêm `'^'` vào bảng ưu tiên với giá trị 3.
- Mức 2: điều kiện pop: `PREC[top] > PREC[c]` nếu `c == '^'`, ngược lại `>=`.
- Mức 3: `2^3^2 = 2^9 = 512`.

## 19. Đáp án tham khảo

In [29]:
# Bài 1
assert Solution().calculate("(6-14)/3*2+1") == -3

# Bài 2
assert Solution().calculate("3*(2-(1+4))") == -9

# Bài 3
def calc_ii_ans(s):
    return calc2(s.replace(' ', ''))
assert calc_ii_ans(" 3+5 / 2 ") == 5 and calc_ii_ans("14-3/2") == 13

# Bài 4
def calc_rd_mod_ans(s):
    s = s.replace(' ', '')
    n, i = len(s), 0
    def expr():
        nonlocal i
        v = term()
        while i < n and s[i] in '+-':
            op = s[i]; i += 1
            w = term()
            v = v + w if op == '+' else v - w
        return v
    def term():
        nonlocal i
        v = factor()
        while i < n and s[i] in '*/%':
            op = s[i]; i += 1
            w = factor()
            if op == '*': v = v * w
            elif op == '/': v = div0(v, w)
            else: v = v - w * div0(v, w)
        return v
    def factor():
        nonlocal i
        if s[i] == '(':
            i += 1; v = expr(); i += 1
            return v
        if s[i] == '-':
            i += 1
            return -factor()
        j = i
        while i < n and s[i].isdigit():
            i += 1
        return int(s[j:i])
    return expr()

assert calc_rd_mod_ans("7%(0-3)") == 1 and calc_rd_mod_ans("(0-7)%3") == -1
assert calc_rd_mod_ans("10%4*3") == 6

# Bài 5
def calc_pow_ans(s):
    prec = {'+': 1, '-': 1, '*': 2, '/': 2, '^': 3}
    nums, ops = [], []
    def app():
        op = ops.pop(); b = nums.pop(); a = nums.pop()
        nums.append(a ** b if op == '^' else
                    a + b if op == '+' else a - b if op == '-' else
                    a * b if op == '*' else div0(a, b))
    for tok in re.findall(r'\d+|[-+*/^()]', s):
        if tok.isdigit():
            nums.append(int(tok))
        elif tok == '(':
            ops.append(tok)
        elif tok == ')':
            while ops[-1] != '(':
                app()
            ops.pop()
        else:
            while ops and ops[-1] != '(' and (prec[ops[-1]] > prec[tok] or
                                              (prec[ops[-1]] == prec[tok] and tok != '^')):
                app()
            ops.append(tok)
    while ops:
        app()
    return nums[-1]

assert calc_pow_ans("2^3^2") == 512 and calc_pow_ans("(2^3)^2") == 64
assert calc_pow_ans("2*3^2-1") == 17
random.seed(3)
for _ in range(300):
    s = rand_expr()
    r = safe(ref_eval, s)
    if r != "div0":
        assert calc_pow_ans(s) == r and calc_rd_mod_ans(s) == r
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (cất trạng thái, không đệ quy):**

```python
saved, stack, num, op = [], [], 0, '+'
for c in s.replace(' ', '') + '+':
    if c.isdigit(): num = num*10 + int(c)
    elif c == '(': saved.append((stack, op)); stack, num, op = [], 0, '+'
    else:
        apply op lên num  (+ push, - push -num, * stack[-1]*=num, / cắt về 0)
        if c == ')': num = sum(stack); stack, op = saved.pop()
        else: op, num = c, 0
return sum(stack)
```

**Template văn phạm:** `expr := term (± term)*`, `term := factor (*/ factor)*`, `factor := số | '(' expr ')' | '-' factor`.

**Checklist trước khi submit:**

- [ ] Chia **cắt về 0**, không dùng `//` trên số âm.
- [ ] Chốt số cuối (`+ '+'` hoặc kiểm tra cuối chuỗi).
- [ ] Số nhiều chữ số.
- [ ] Ngoặc: cất/khôi phục **cả** `stack` và `op`.
- [ ] Kết hợp trái: `8-3-2 = 3`, `8/3/2 = 1`.
- [ ] Bỏ dấu cách (nếu có).
- [ ] Ngoặc lồng sâu → tránh đệ quy hoặc tăng recursion limit.
- [ ] Tên method đúng: `calculate(self, s)`.